# FakeBDTeen Fusion Trainer (Notebook 2)
This notebook loads the cached embeddings, trains the cross-attention model, and evaluates multi-label performance. Run after Notebook 1.

## 1. Dependencies & Feature Extraction
Import libraries, extract the feature ZIP, and load metadata.

In [ ]:
import os
import zipfile
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score, hamming_loss, roc_auc_score, roc_curve
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

dataset_root = '/kaggle/input/datasets/tanmoykdas/fakebdteen-extracted-feature-dataset/fakebdteen_extracted_features'
features_root = '/kaggle/working/'

os.makedirs(features_root, exist_ok=True)

def find_feature_root(search_root: str):
    for root, dirs, files in os.walk(search_root):
        if 'audio' in dirs and 'video' in dirs and 'fakebdteen_metadata.csv' in files:
            return root
    return None

features_zip = os.path.join(dataset_root, 'fakebdteen_extracted_features.zip')
if os.path.exists(features_zip):
    with zipfile.ZipFile(features_zip, 'r') as zf:
        zf.extractall('/kaggle/working')
    candidate_root = find_feature_root('/kaggle/working')
else:
    candidate_root = find_feature_root(dataset_root)

if candidate_root is None:
    raise FileNotFoundError(
        'Feature folders not found. Expected a folder containing audio/, video/, and fakebdteen_metadata.csv.'
    )

features_root = candidate_root
metadata_csv = os.path.join(features_root, 'fakebdteen_metadata.csv')

if not os.path.exists(metadata_csv):
    raise FileNotFoundError('fakebdteen_metadata.csv not found in features_root.')

registry_df = pd.read_csv(metadata_csv)
print(registry_df.head())
print(f'Total records: {len(registry_df)}')
print(f'Using features_root: {features_root}')

## 2. Speaker-Independent Split
Use GroupKFold to prevent speaker leakage between train and validation.

In [ ]:
gkf = GroupKFold(n_splits=5)
groups = registry_df['subject_id'].astype(str).values
splits = list(gkf.split(registry_df, registry_df[['video_label', 'audio_label']], groups))
train_idx, val_idx = splits[0]
train_df = registry_df.iloc[train_idx].reset_index(drop=True)
val_df = registry_df.iloc[val_idx].reset_index(drop=True)

train_speakers = sorted(train_df['subject_id'].unique().tolist())
val_speakers = sorted(val_df['subject_id'].unique().tolist())

print(f'Train speakers ({len(train_speakers)}): {train_speakers}')
print(f'Val speakers ({len(val_speakers)}): {val_speakers}')
print(f'Overlap: {set(train_speakers).intersection(set(val_speakers))}')

## 3. Dataset & DataLoaders
Load cached .npy features on the fly with a custom Dataset.

In [ ]:
class SavedFeaturesDataset(Dataset):
    def __init__(self, df: pd.DataFrame, features_root: str):
        self.df = df
        self.features_root = features_root
        self.audio_root = os.path.join(features_root, 'audio')
        self.video_root = os.path.join(features_root, 'video')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        unique_id = row['unique_id']
        audio_path = os.path.join(self.audio_root, f'{unique_id}.npy')
        video_path = os.path.join(self.video_root, f'{unique_id}.npy')
        audio_feat = np.load(audio_path).astype(np.float32)
        video_feat = np.load(video_path).astype(np.float32)
        audio_feat = torch.from_numpy(audio_feat)
        video_feat = torch.from_numpy(video_feat)
        video_label = torch.tensor(row['video_label'], dtype=torch.float32)
        audio_label = torch.tensor(row['audio_label'], dtype=torch.float32)
        return video_feat, audio_feat, video_label, audio_label

train_dataset = SavedFeaturesDataset(train_df, features_root)
val_dataset = SavedFeaturesDataset(val_df, features_root)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    pin_memory=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    pin_memory=True,
    num_workers=0
)

## 4. Cross-Attention Model
Define the fusion architecture and dual binary heads.

In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, embed_dim: int = 512, num_heads: int = 8):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True, dropout=0.1)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, 2048),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(2048, embed_dim)
        )

    def forward(self, q, k, v):
        attn_out, _ = self.attn(q, k, v, need_weights=False)
        q = self.norm1(q + attn_out)
        ff_out = self.ff(q)
        return self.norm2(q + ff_out)

class MultiModalDeepfakeDetector(nn.Module):
    def __init__(self, video_dim: int = 2048, audio_dim: int = 1024, embed_dim: int = 512):
        super().__init__()
        self.video_proj = nn.Sequential(
            nn.Linear(video_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.audio_proj = nn.Sequential(
            nn.Linear(audio_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.cross_attn_v2a = CrossAttention(embed_dim=embed_dim, num_heads=8)
        self.cross_attn_a2v = CrossAttention(embed_dim=embed_dim, num_heads=8)
        self.fusion_layer = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.video_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        self.audio_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        self.fusion_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, video_feat, audio_feat):
        video_emb = self.video_proj(video_feat)
        audio_emb = self.audio_proj(audio_feat)
        video_fused = self.cross_attn_v2a(video_emb, audio_emb, audio_emb)
        audio_fused = self.cross_attn_a2v(audio_emb, video_emb, video_emb)
        fused = self.fusion_layer(torch.cat([video_fused.mean(dim=1), audio_fused.mean(dim=1)], dim=1))
        video_logit = self.video_head(fused).squeeze(-1)
        audio_logit = self.audio_head(fused).squeeze(-1)
        fusion_logit = self.fusion_head(fused).squeeze(-1)
        return video_logit, audio_logit, fusion_logit

## 5. Training
Train the multi-task model with AMP on GPU.

In [ ]:
import contextlib

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiModalDeepfakeDetector().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else torch.amp.GradScaler()
amp_ctx = torch.amp.autocast('cuda') if device.type == 'cuda' else contextlib.nullcontext()

epochs = 15
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for video_feat, audio_feat, video_label, audio_label in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}'):
        video_feat = video_feat.to(device, non_blocking=True)
        audio_feat = audio_feat.to(device, non_blocking=True)
        video_label = video_label.to(device, non_blocking=True)
        audio_label = audio_label.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with amp_ctx:
            video_logit, audio_logit, fusion_logit = model(video_feat, audio_feat)
            loss_video = criterion(video_logit, video_label)
            loss_audio = criterion(audio_logit, audio_label)
            loss_fusion = criterion(fusion_logit, (video_label + audio_label) / 2)
            loss = 0.35 * loss_video + 0.35 * loss_audio + 0.3 * loss_fusion
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * video_feat.size(0)
    avg_loss = running_loss / len(train_loader.dataset)
    print(f'Epoch {epoch} | Train Loss: {avg_loss:.6f}')

## 6. Multi-Label Evaluation
Compute multi-label metrics and per-head classification reports.

In [ ]:
model.eval()
all_video_logits = []
all_audio_logits = []
all_fusion_logits = []
all_video_labels = []
all_audio_labels = []

with torch.no_grad():
    for video_feat, audio_feat, video_label, audio_label in tqdm(val_loader, desc='Validation'):
        video_feat = video_feat.to(device, non_blocking=True)
        audio_feat = audio_feat.to(device, non_blocking=True)
        video_logit, audio_logit, fusion_logit = model(video_feat, audio_feat)
        all_video_logits.append(video_logit.cpu())
        all_audio_logits.append(audio_logit.cpu())
        all_fusion_logits.append(fusion_logit.cpu())
        all_video_labels.append(video_label.cpu())
        all_audio_labels.append(audio_label.cpu())

video_logits = torch.cat(all_video_logits).numpy()
audio_logits = torch.cat(all_audio_logits).numpy()
fusion_logits = torch.cat(all_fusion_logits).numpy()
video_labels = torch.cat(all_video_labels).numpy()
audio_labels = torch.cat(all_audio_labels).numpy()

video_probs = 1 / (1 + np.exp(-video_logits))
audio_probs = 1 / (1 + np.exp(-audio_logits))
fusion_probs = 1 / (1 + np.exp(-fusion_logits))

video_preds = (video_probs >= 0.5).astype(int)
audio_preds = (audio_probs >= 0.5).astype(int)
fusion_preds = (fusion_probs >= 0.5).astype(int)

video_acc = accuracy_score(video_labels, video_preds)
audio_acc = accuracy_score(audio_labels, audio_preds)
fusion_acc = accuracy_score((video_labels + audio_labels) / 2 >= 0.5, fusion_preds)

video_f1 = f1_score(video_labels, video_preds)
audio_f1 = f1_score(audio_labels, audio_preds)
fusion_f1 = f1_score((video_labels + audio_labels) / 2 >= 0.5, fusion_preds)

try:
    video_auc = roc_auc_score(video_labels, video_probs)
    audio_auc = roc_auc_score(audio_labels, audio_probs)
    fusion_auc = roc_auc_score((video_labels + audio_labels) / 2 >= 0.5, fusion_probs)
except:
    video_auc = audio_auc = fusion_auc = 0.0

print('='*60)
print('VIDEO HEAD PERFORMANCE')
print('='*60)
print(f'Accuracy: {video_acc:.4f}')
print(f'F1-Score: {video_f1:.4f}')
print(f'AUC-ROC: {video_auc:.4f}')
print(classification_report(video_labels, video_preds, target_names=['Real', 'Fake']))

print('='*60)
print('AUDIO HEAD PERFORMANCE')
print('='*60)
print(f'Accuracy: {audio_acc:.4f}')
print(f'F1-Score: {audio_f1:.4f}')
print(f'AUC-ROC: {audio_auc:.4f}')
print(classification_report(audio_labels, audio_preds, target_names=['Real', 'Fake']))

print('='*60)
print('FUSION HEAD PERFORMANCE')
print('='*60)
print(f'Accuracy: {fusion_acc:.4f}')
print(f'F1-Score: {fusion_f1:.4f}')
print(f'AUC-ROC: {fusion_auc:.4f}')
print(classification_report((video_labels + audio_labels) / 2 >= 0.5, fusion_preds, target_names=['Real', 'Fake']))

## 6.1 Fusion Behavior Visualization
Show confusion matrices, fusion probability separation, and audio-video interaction.

In [ ]:
from sklearn.metrics import confusion_matrix

fusion_true = ((video_labels + audio_labels) / 2 >= 0.5).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
cm_video = confusion_matrix(video_labels.astype(int), video_preds.astype(int))
cm_audio = confusion_matrix(audio_labels.astype(int), audio_preds.astype(int))
cm_fusion = confusion_matrix(fusion_true, fusion_preds.astype(int))

for ax, cm, title in zip(
    axes,
    [cm_video, cm_audio, cm_fusion],
    ['Video Head Confusion Matrix', 'Audio Head Confusion Matrix', 'Fusion Head Confusion Matrix']
 ):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)

plt.tight_layout()
plt.show()

# Fusion probability behavior
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(fusion_probs[fusion_true == 0], bins=20, alpha=0.7, label='True Real', color='#4e79a7')
axes[0].hist(fusion_probs[fusion_true == 1], bins=20, alpha=0.7, label='True Fake', color='#e15759')
axes[0].set_title('Fusion Probability Separation')
axes[0].set_xlabel('Fusion Probability')
axes[0].set_ylabel('Count')
axes[0].legend()

scatter = axes[1].scatter(video_probs, audio_probs, c=fusion_true, cmap='coolwarm', alpha=0.6)
axes[1].set_title('Fusion Input Space (Video vs Audio Prob)')
axes[1].set_xlabel('Video Probability')
axes[1].set_ylabel('Audio Probability')
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('Fusion Ground Truth (0=Real, 1=Fake)')

plt.tight_layout()
plt.show()

## 7. Visualization & Results Export
Generate performance plots and export results to ZIP.

In [ ]:
results_dir = '/kaggle/working/evaluation_results'
os.makedirs(results_dir, exist_ok=True)

metrics_data = {
    'Modality': ['Video', 'Audio', 'Fusion'],
    'Accuracy': [video_acc, audio_acc, fusion_acc],
    'F1-Score': [video_f1, audio_f1, fusion_f1],
    'AUC-ROC': [video_auc, audio_auc, fusion_auc]
}
metrics_df = pd.DataFrame(metrics_data)
metrics_df.to_csv(os.path.join(results_dir, 'performance_metrics.csv'), index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('FakeBDTeen Multi-Modal Deepfake Detection - Performance Comparison', fontsize=14, fontweight='bold')

metrics = ['Accuracy', 'F1-Score', 'AUC-ROC']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for idx, metric in enumerate(metrics):
    values = [video_acc, audio_acc, fusion_acc] if metric == 'Accuracy' else [video_f1, audio_f1, fusion_f1] if metric == 'F1-Score' else [video_auc, audio_auc, fusion_auc]
    axes[idx].bar(['Video', 'Audio', 'Fusion'], values, color=colors)
    axes[idx].set_ylabel('Score', fontweight='bold')
    axes[idx].set_title(metric, fontweight='bold')
    axes[idx].set_ylim([0, 1])
    for i, v in enumerate(values):
        axes[idx].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'performance_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('ROC Curves - Per Modality', fontsize=14, fontweight='bold')

try:
    fpr_v, tpr_v, _ = roc_curve(video_labels, video_probs)
    axes[0, 0].plot(fpr_v, tpr_v, linewidth=2, label=f'AUC = {video_auc:.4f}')
    axes[0, 0].plot([0, 1], [0, 1], 'k--', linewidth=1)
    axes[0, 0].set_xlabel('False Positive Rate')
    axes[0, 0].set_ylabel('True Positive Rate')
    axes[0, 0].set_title('Video Modality', fontweight='bold')
    axes[0, 0].legend(loc='lower right')
    axes[0, 0].grid(alpha=0.3)

    fpr_a, tpr_a, _ = roc_curve(audio_labels, audio_probs)
    axes[0, 1].plot(fpr_a, tpr_a, linewidth=2, color='#4ECDC4', label=f'AUC = {audio_auc:.4f}')
    axes[0, 1].plot([0, 1], [0, 1], 'k--', linewidth=1)
    axes[0, 1].set_xlabel('False Positive Rate')
    axes[0, 1].set_ylabel('True Positive Rate')
    axes[0, 1].set_title('Audio Modality', fontweight='bold')
    axes[0, 1].legend(loc='lower right')
    axes[0, 1].grid(alpha=0.3)

    fusion_labels_binary = (video_labels + audio_labels) / 2 >= 0.5
    fpr_f, tpr_f, _ = roc_curve(fusion_labels_binary, fusion_probs)
    axes[1, 0].plot(fpr_f, tpr_f, linewidth=2, color='#45B7D1', label=f'AUC = {fusion_auc:.4f}')
    axes[1, 0].plot([0, 1], [0, 1], 'k--', linewidth=1)
    axes[1, 0].set_xlabel('False Positive Rate')
    axes[1, 0].set_ylabel('True Positive Rate')
    axes[1, 0].set_title('Fusion (Combined)', fontweight='bold')
    axes[1, 0].legend(loc='lower right')
    axes[1, 0].grid(alpha=0.3)
except Exception as e:
    print(f'Warning: Could not plot ROC curves: {e}')

axes[1, 1].axis('off')
summary_text = f"""EVALUATION SUMMARY
Dataset Split: Speaker-Independent (GroupKFold)
Training Samples: {len(train_df)}
Validation Samples: {len(val_df)}

Video Head: Acc={video_acc:.4f}, F1={video_f1:.4f}
Audio Head: Acc={audio_acc:.4f}, F1={audio_f1:.4f}
Fusion Head: Acc={fusion_acc:.4f}, F1={fusion_f1:.4f}"""
axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace', verticalalignment='center')

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Performance and ROC plots saved to evaluation_results/')

In [ ]:
predictions_data = {
    'video_pred': video_preds,
    'video_prob': video_probs,
    'video_true': video_labels.astype(int),
    'audio_pred': audio_preds,
    'audio_prob': audio_probs,
    'audio_true': audio_labels.astype(int),
    'fusion_pred': fusion_preds,
    'fusion_prob': fusion_probs,
    'fusion_true': (video_labels + audio_labels) / 2 >= 0.5
}
predictions_df = pd.DataFrame(predictions_data)
predictions_df.to_csv(os.path.join(results_dir, 'predictions.csv'), index=False)

summary_report = f"""FakeBDTeen Multimodal Deepfake Detection - Evaluation Report
='*80

Dataset Configuration:
  Total Samples: {len(registry_df)}
  Training Samples: {len(train_df)}
  Validation Samples: {len(val_df)}
  Speaker-Independent Split: Yes

Model Architecture:
  Video Encoder: ResNet50 (2048D features)
  Audio Encoder: Wav2Vec2 XLS-R (1024D features)
  Fusion: Cross-Attention + Multi-Head Classification
  Embedding Dimension: 512
  Attention Heads: 8

Training Configuration:
  Epochs: 15
  Batch Size: 64
  Optimizer: AdamW (lr=1e-4)
  Loss: Weighted BCE (Video:0.35, Audio:0.35, Fusion:0.3)

Performance Metrics:
  VIDEO HEAD:
    Accuracy: {video_acc:.4f}
    F1-Score: {video_f1:.4f}
    AUC-ROC: {video_auc:.4f}

  AUDIO HEAD:
    Accuracy: {audio_acc:.4f}
    F1-Score: {audio_f1:.4f}
    AUC-ROC: {audio_auc:.4f}

  FUSION HEAD:
    Accuracy: {fusion_acc:.4f}
    F1-Score: {fusion_f1:.4f}
    AUC-ROC: {fusion_auc:.4f}

Key Findings:
  - Audio modality shows significantly higher accuracy ({audio_acc:.4f}) vs video ({video_acc:.4f})
  - Fusion model combines both modalities for improved generalization
  - Speaker-independent evaluation ensures no data leakage

Output Files:
  - performance_metrics.csv: Comparison of metrics across modalities
  - performance_comparison.png: Bar charts of accuracy, F1, AUC
  - roc_curves.png: ROC curves for each modality
  - predictions.csv: Detailed predictions on validation set
  - evaluation_report.txt: This summary report
"""

with open(os.path.join(results_dir, 'evaluation_report.txt'), 'w') as f:
    f.write(summary_report)

print('Summary report generated!')
print(summary_report)

In [ ]:
zip_output = '/kaggle/working/FakeBDTeen_Evaluation_Results.zip'
if os.path.exists(zip_output):
    os.remove(zip_output)

with zipfile.ZipFile(zip_output, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(results_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, results_dir)
            zf.write(file_path, arcname)

print(f'✓ Evaluation results packaged into: {zip_output}')
print(f'✓ ZIP size: {os.path.getsize(zip_output) / (1024*1024):.2f} MB')
print(f'✓ Contents:')
print(f'  - performance_metrics.csv')
print(f'  - performance_comparison.png')
print(f'  - roc_curves.png')
print(f'  - predictions.csv')
print(f'  - evaluation_report.txt')
print(f'\nReady for download!')